# DCASE 2026 Task 1 — projektni izveštaj

**Tema:** Heterogena klasifikacija zvuka pomoću Broad Sound Taxonomy (BST)

**Korišćeno rešenje:** zvanični baseline za DCASE 2026 Task 1

Ovaj notebook dokumentuje cilj projekta, osposobljavanje baseline sistema, izvedene eksperimente i rezultate ostvarene na skupu BSD10k-v1.2. Posebno se ispituje uticaj pouzdanosti anotacija na klasifikaciju zvuka.

## 1. Cilj projekta i istraživačko pitanje

DCASE 2026 Task 1 se bavi sistemom koji proizvoljan zvučni snimak svrstava u široku kategoriju zvuka. Klasifikacija se vrši prema hijerarhijskoj taksonomiji BST, koja sadrži 5 kategorija najvišeg nivoa i 23 kategorije drugog nivoa.

Glavno istraživačko pitanje projekta glasi:

> Da li manji, ali pouzdanije anotiran skup daje bolji rezultat od kompletnog skupa?

Zvanični baseline je najpre reprodukovan i proveren, čime je obezbeđena pouzdana referentna tačka za poređenje različitih varijanti skupa podataka.

## 2. Podaci i reprezentacija

U eksperimentima je korišćen **BSD10k-v1.2**, kurirani Freesound skup sa ekspertskim anotacijama. Pripremljeno je 10.956 primera. Za svaki primer postoje dve unapred izračunate reprezentacije dimenzije 512:

- audio embedding dobijen modelom LAION-CLAP;
- tekstualni embedding opisa zvuka, takođe dobijen modelom LAION-CLAP.

Skripta `build_dataset.py` povezuje metapodatke, putanje embeddinga, oznake kategorija i njihove numeričke indekse. Generisani su `processed_dataset.csv` i JSON rečnici kategorija. Provereno je da za svih 10.956 primera postoje i audio i tekstualni embedding.

In [ ]:
from pathlib import Path
import csv

PROJECT_DIR = Path.cwd()
BASELINE_DIR = PROJECT_DIR / "dcase2026_task1_baseline"
if not BASELINE_DIR.exists():
    BASELINE_DIR = PROJECT_DIR

dataset_csv = BASELINE_DIR / "data" / "processed_dataset.csv"
with dataset_csv.open(encoding="utf-8") as file:
    reader = csv.DictReader(file)
    rows = list(reader)

print(f"Broj pripremljenih primera: {len(rows):,}".replace(",", "."))
print(f"Broj kategorija drugog nivoa: {len({row['class'] for row in rows})}")
print(f"Broj kategorija najvišeg nivoa: {len({row['top_class'] for row in rows})}")

## 3. Baseline model

Baseline je varijanta HATR modela i podržava dva režima:

1. **Audio-only** — klasifikacija samo na osnovu audio embeddinga.
2. **Multimodalni (`both`)** — koristi audio i tekstualni embedding.

Model sadrži odvojene enkodere modaliteta, mehanizam pažnje za njihovu fuziju i klasifikator sa rezidualnim blokovima. Tokom treniranja koriste se augmentacije embeddinga: Gausov šum i nasumično maskiranje. Model direktno predviđa jednu od 23 kategorije drugog nivoa, dok se pripadnost kategoriji najvišeg nivoa izvodi iz BST hijerarhije.

Za evaluaciju su korišćene standardne i hijerarhijske metrike. Najvažnija metrika zadatka je **hierarchical F1**, jer uzima u obzir i predikciju konkretnije kategorije i njen položaj u hijerarhiji.

## 4. Osposobljavanje i izmene

Urađeno je sledeće:

- kreirano je lokalno Python okruženje i instalirane su potrebne biblioteke;
- preuzeti su metapodaci i CLAP embedding reprezentacije za BSD10k-v1.2;
- uspešno je izvršena priprema skupa;
- potvrđeno je da kompletan tok radi: učitavanje podataka, treniranje, izbor najboljeg checkpointa, testiranje i čuvanje predikcija;
- `train_test.py` je prilagođen tako da se režimi, izlazni direktorijum, broj epoha, maksimalan broj foldova i broj DataLoader procesa mogu zadati promenljivama okruženja;
- time je omogućeno bezbedno izvođenje kratkih provera bez menjanja glavnih hiperparametara u kodu.

Primer komande za kontrolisani eksperiment:

```bash
DCASE_MODES=audio,both \
DCASE_MODEL_OUTPUT=./model_output_medium \
DCASE_NUM_EPOCHS=10 \
DCASE_K_FOLDS=5 \
DCASE_MAX_FOLDS=1 \
DCASE_NUM_WORKERS=0 \
.venv/bin/python train_test.py
```

## 5. Provera eksperimentalnog toka

Pre punog treninga izvedena su dva nivoa provere.

### 5.1. Smoke test

Oba režima su trenirana dve epohe na jednom foldu. Cilj nije bio ostvarivanje konačnog rezultata, već potvrda da ceo pipeline radi.

| Režim | Accuracy | Hierarchical accuracy | Hierarchical F1 |
|---|---:|---:|---:|
| Audio | 64,87% | 60,31% | 65,44% |
| Audio + tekst | 72,67% | 68,58% | 69,07% |

### 5.2. Srednji test

Zatim su oba režima trenirana 10 epoha na jednom foldu. Ovaj test je potvrdio stabilan rast validacione tačnosti i pravilno čuvanje najboljeg modela.

| Režim | Accuracy | Macro accuracy | Hierarchical accuracy | Hierarchical F1 |
|---|---:|---:|---:|---:|
| Audio | 75,96% | 64,83% | 71,49% | 72,24% |
| Audio + tekst | 77,69% | 66,75% | 72,98% | 72,67% |

Najbolja validaciona tačnost za oba modela ostvarena je u osmoj epohi: 76,16% za audio i 78,61% za multimodalni model.

## 6. Puni 5-fold eksperiment

Nakon provere eksperimentalnog toka pokrenut je puni eksperiment za oba režima. Korišćeno je stratifikovano 5-fold unakrsno testiranje. U svakom foldu približno 20% podataka čini test skup, a deo preostalih podataka je izdvojen za validaciju.

Maksimalan broj epoha bio je 100, uz early stopping. Zbog toga su pojedinačni treninzi završeni između 30. i 49. epohe za audio, odnosno između 32. i 42. epohe za multimodalni režim. Za svaki fold sačuvani su:

- najbolji model (`best_model.pth`);
- istorija treniranja (`history.json`);
- podela primera (`splits.csv`);
- test predikcije (`predictions.csv`);
- evaluacione metrike (`results.txt`).

In [ ]:
import re
from collections import defaultdict
from statistics import mean, pstdev

results_root = BASELINE_DIR / "model_output_full"
pattern = re.compile(r"([a-zA-Z0-9_]+):\s*([\d.]+)%")
metrics = defaultdict(lambda: defaultdict(list))

for result_file in sorted(results_root.glob("*/fold_*/evaluation/results.txt")):
    mode = result_file.parts[-4]
    for name, value in pattern.findall(result_file.read_text(encoding="utf-8")):
        metrics[mode][name].append(float(value))

for mode in ("audio", "both"):
    print(f"{mode}:")
    for metric in ("accuracy", "macro_accuracy", "hierarchical_accuracy", "hierarchical_f1"):
        values = metrics[mode][metric]
        print(f"  {metric}: {mean(values):.2f}% ± {pstdev(values):.2f}%")

### 6.1. Zbirni rezultati

| Režim | Accuracy | Macro accuracy | Top-level accuracy | Hierarchical accuracy | Hierarchical F1 |
|---|---:|---:|---:|---:|---:|
| Audio | 77,09% ± 0,64% | 69,88% ± 1,36% | 87,71% ± 0,28% | 76,49% ± 0,96% | 75,20% ± 0,90% |
| Audio + tekst | **79,98% ± 0,66%** | **74,32% ± 0,77%** | **89,18% ± 0,68%** | **79,66% ± 0,75%** | **78,65% ± 0,61%** |

Multimodalni model je u odnosu na audio-only model ostvario poboljšanje od:

- 2,89 procentnih poena za accuracy;
- 4,44 procentna poena za macro accuracy;
- 3,17 procentnih poena za hierarchical accuracy;
- **3,45 procentnih poena za hierarchical F1**.

Manja standardna devijacija hierarchical F1 multimodalnog modela (0,61 naspram 0,90) ukazuje i na nešto stabilnije rezultate između foldova.

In [ ]:
# Rezultat hierarchical F1 po foldovima
for mode in ("audio", "both"):
    values = metrics[mode]["hierarchical_f1"]
    formatted = ", ".join(f"{value:.2f}%" for value in values)
    print(f"{mode}: {formatted}")

### 6.2. Poređenje sa objavljenim baseline rezultatima

| Režim | Naš hierarchical accuracy | Objavljeno | Naš hierarchical F1 | Objavljeno |
|---|---:|---:|---:|---:|
| Audio | 76,49% | 77,36% | 75,20% | 76,11% |
| Audio + tekst | 79,66% | 79,71% | 78,65% | 78,76% |

Multimodalni rezultat je gotovo identičan objavljenom rezultatu: razlika je 0,05 poena za hierarchical accuracy i 0,11 poena za hierarchical F1. Audio rezultat je niži za približno 0,9 poena, ali prati očekivani opseg i potvrđuje da je baseline uspešno reprodukovan. Manje razlike mogu nastati zbog hardvera, verzija biblioteka i nedeterminističkih numeričkih operacija.

## 7. Eksperiment sa pouzdanijim anotacijama

Za pouzdaniji podskup izabrani su primeri sa `confidence >= 4`. Analiza mogućih pragova pokazala je da ovaj kriterijum predstavlja dobar kompromis između kvaliteta i veličine: zadržano je 6.820 od 10.956 primera (62,25%), svih 23 kategorija, a najmanja kategorija i dalje sadrži 72 primera. Prag 5 bi ostavio samo 777 primera i izgubio jednu kategoriju.

Filtrirani skup napravljen je reproduktibilnom skriptom `prepare_confident_subset.py`. Raspodela nije ostala potpuno ista: procenat zadržanih primera po klasi kreće se od 30,30% do 86,74%, što je važno pri tumačenju makro metrika. Nad filtriranim skupom ponovljen je isti 5-fold eksperiment za oba režima, sa istim hiperparametrima i early stopping procedurom.

| Skup | Režim | Accuracy | Macro accuracy | Hierarchical accuracy | Hierarchical F1 |
|---|---|---:|---:|---:|---:|
| Kompletan (10.956) | Audio | 77,09% ± 0,64% | 69,88% ± 1,36% | 76,49% ± 0,96% | 75,20% ± 0,90% |
| Pouzdaniji (6.820) | Audio | **85,62% ± 1,04%** | **75,79% ± 2,01%** | **81,21% ± 1,53%** | **80,06% ± 3,02%** |
| Kompletan (10.956) | Audio + tekst | 79,98% ± 0,66% | 74,32% ± 0,77% | 79,66% ± 0,75% | 78,65% ± 0,61% |
| Pouzdaniji (6.820) | Audio + tekst | **87,86% ± 0,38%** | **79,64% ± 0,73%** | **83,86% ± 0,63%** | **83,01% ± 0,47%** |

Na filtriranom skupu hierarchical F1 je veći za 4,86 poena u audio režimu i 4,36 poena u multimodalnom režimu. Multimodalni filtrirani model je posebno stabilan između foldova. Audio režim ima veću standardnu devijaciju zbog jednog slabijeg folda (74,38%), što ukazuje na osetljivost ređih i neujednačeno filtriranih klasa.

**Ograničenje ovog poređenja:** modeli nad kompletnim i filtriranim skupom nisu evaluirani na identičnim test primerima. Filtrirana evaluacija sadrži samo pouzdanije anotacije i zato može biti sama po sebi lakša. Zbog toga je izveden i kontrolisani eksperiment sa zajedničkim test skupom, opisan u sledećem odeljku.

## 8. Kontrolisani eksperiment sa zajedničkim test skupom

Izdvojen je jedan stratifikovani test skup od 2.192 primera (20% kompletnog skupa). Isti zaključani test skup korišćen je za svih šest modela i nije učestvovao u treniranju niti izboru checkpointa. Poređene su tri trening varijante: kompletan preostali skup (7.011 train + 1.753 validation), `confidence >= 4` (4.320 + 1.081) i nasumični podskup iste veličine i klasne raspodele kao pouzdani podskup.

| Trening skup | Režim | Accuracy | Macro accuracy | Hierarchical accuracy | Hierarchical F1 |
|---|---|---:|---:|---:|---:|
| Kompletan | Audio | **78,19%** | **72,18%** | **78,10%** | **76,83%** |
| `confidence >= 4` | Audio | 78,01% | 71,17% | 77,68% | 75,91% |
| Nasumični podskup | Audio | 76,19% | 66,57% | 73,48% | 71,95% |
| Kompletan | Audio + tekst | **80,98%** | **75,82%** | **81,04%** | **79,78%** |
| `confidence >= 4` | Audio + tekst | 78,88% | 72,69% | 78,91% | 77,11% |
| Nasumični podskup | Audio + tekst | 80,16% | 73,44% | 79,06% | 77,63% |

Kontrolisani rezultati pokazuju da manji pouzdani skup **ne nadmašuje kompletan skup** na zajedničkom testu. U audio režimu zaostaje 0,92 poena hierarchical F1, a u multimodalnom 2,67 poena. Ipak, pouzdani audio podskup nadmašuje nasumični podskup iste veličine za 3,96 poena, pa kvalitetan izbor anotacija ublažava gubitak izazvan smanjenjem količine podataka. Kod multimodalnog modela nasumični podskup je 0,52 poena bolji od pouzdanog.

Zaključak istraživačkog pitanja je nijansiran: pouzdanije anotacije mogu biti vrednije od nasumično izabrane iste količine podataka, naročito za audio-only model, ali dodatna količina podataka iz kompletnog skupa ipak daje najbolju generalizaciju.

## 9. Zaključci

1. Zvanični DCASE 2026 Task 1 baseline je uspešno osposobljen i reprodukovan na BSD10k-v1.2 skupu.
2. Ceo tok od pripreme podataka do evaluacije radi za audio i multimodalni režim.
3. Tekstualna reprezentacija opisa zvuka daje jasno i konzistentno poboljšanje.
4. Multimodalni model postiže hierarchical F1 od 78,65% ± 0,61%, odnosno 3,45 procentnih poena više od audio-only modela.
5. Ostvareni multimodalni rezultat veoma je blizak zvanično objavljenom baseline rezultatu.
6. Podskup sa `confidence >= 4` sadrži 6.820 primera i zadržava sve 23 kategorije.
7. Viši nekontrolisani rezultat filtriranog skupa delom nastaje zato što je i njegov test deo filtriran i lakši.
8. Na zajedničkom test skupu kompletan trening skup daje najbolji hierarchical F1: 76,83% za audio i 79,78% za multimodalni model.
9. Pouzdani audio podskup je za 3,96 poena bolji od nasumičnog podskupa iste veličine, ali za 0,92 poena slabiji od kompletnog trening skupa.

Kontrolisani eksperiment ne potvrđuje da manji pouzdani skup nadmašuje kompletan skup. Potvrđuje, međutim, da kvalitetan izbor primera može biti znatno bolji od nasumičnog smanjenja iste veličine u audio režimu.